# Treino RF e GB por área

Treina RandomForest e GradientBoosting por componente em cada uma das sete áreas, nas duas versões de perdas (`_perfil` e `_13cat`), avaliando apenas nas 17 dobras leave-one-group-out da partição gravada. As previsões ficam guardadas para os notebooks seguintes.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_treinamento"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os, json, warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning, module=r"sklearn|joblib")   # avisos de paralelismo

SUFIXOS = ("_perfil", "_13cat")
PASTA_DADOS    = "colabs_S3"
PASTA_PARTICAO = "colabs_S3/preparacao_testes"

BASES = [
    ".", "..", os.path.join("..", ".."), os.path.join("..", "..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
    os.path.join("notebooks", PASTA_PARTICAO),
]

def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )

In [ ]:
AREAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
         "main_vent_sys", "mainback", "oven_hmi"]

datasets = {
    area: pd.read_csv(resolve(PASTA_DADOS, f"dataset_{area}{SUFIXOS[0]}.csv"))
    for area in AREAS
}

limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv")).set_index("tc_name")

In [ ]:
with open(resolve(PASTA_PARTICAO, "particao_treino_teste.json"), encoding="utf-8") as f:
    particao = json.load(f)

IDX_TREINO = particao["run_idx_treino"]
IDX_TESTE  = particao["run_idx_teste"]
DOBRA  = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})

In [ ]:
pd.DataFrame(
    {area: {"linhas": df.shape[0], "colunas": df.shape[1]} for area, df in datasets.items()}
).T.assign(limites=limites.shape[0], treino=len(IDX_TREINO), teste=len(IDX_TESTE), dobras=len(DOBRAS))

In [ ]:
PREFIXOS_FEATURES = [
    "blower_direction", "housing", "board_position", "run_case",
    "losses", "perfil_perdas", "deflector_hole", "material_coil_support",
    "cfg",   # sem este prefixo, os atributos fisicos seriam tratados como alvo
]
COLUNAS_AUXILIARES = ["run_idx"]

def separar(df, prefixos=PREFIXOS_FEATURES):
    """Devolve (X, cols_alvo); run_idx fica fora de ambos."""
    cols_entrada = [c for c in df.columns if any(c.startswith(p + "_") for p in prefixos)]
    cols_alvo    = [c for c in df.columns if c not in cols_entrada + COLUNAS_AUXILIARES]
    return df[cols_entrada].copy(), cols_alvo

In [ ]:
separado = {}
for area, df in datasets.items():
    X, cols_alvo = separar(df)
    separado[area] = {"X": X, "cols_alvo": cols_alvo}

In [ ]:
pd.DataFrame(
    {area: {"features": s["X"].shape[1], "alvos": len(s["cols_alvo"])} for area, s in separado.items()}
).T

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def indices_da_dobra(df, dobra, y_col):
    """(idx_treino, idx_validacao) posicionais; o teste reservado nunca entra."""
    run = df["run_idx"].to_numpy() if "run_idx" in df.columns else np.arange(len(df))
    medido    = y_col.notna().to_numpy()
    no_treino = np.array([r in DOBRA and DOBRA[r] is not None for r in run])
    idx_val = [i for i in range(len(df)) if medido[i] and no_treino[i] and DOBRA[run[i]] == dobra]
    idx_tr  = [i for i in range(len(df)) if medido[i] and no_treino[i] and DOBRA[run[i]] != dobra]
    return idx_tr, idx_val

In [ ]:
HIPERPARAMETROS = {
    "rf": {"n_estimators": 300, "random_state": 42, "n_jobs": -1},
    "gb": {"random_state": 42},
}

def metricas(real, previsto):
    return {"MAE": mean_absolute_error(real, previsto),
            "RMSE": root_mean_squared_error(real, previsto),
            "R2": r2_score(real, previsto)}

def treinar_area(area, hiperparametros=HIPERPARAMETROS):
    """Previsoes fora-da-dobra de RF e GB para cada componente da area, com MAE, RMSE e R2."""
    df, X, cols_alvo = datasets[area], separado[area]["X"], separado[area]["cols_alvo"]
    run = df["run_idx"].to_numpy() if "run_idx" in df.columns else np.arange(len(df))
    previsoes = {}
    for col in cols_alvo:
        y = df[col]
        real, rf, gb, ensaios = [], [], [], []
        for d in DOBRAS:
            idx_tr, idx_val = indices_da_dobra(df, d, y)
            if not idx_tr or not idx_val:
                continue
            X_tr, y_tr, X_val = X.iloc[idx_tr], y.iloc[idx_tr], X.iloc[idx_val]
            m_rf = RandomForestRegressor(**hiperparametros["rf"]).fit(X_tr, y_tr)
            m_gb = GradientBoostingRegressor(**hiperparametros["gb"]).fit(X_tr, y_tr)
            real.extend(y.iloc[idx_val])
            rf.extend(m_rf.predict(X_val))
            gb.extend(m_gb.predict(X_val))
            ensaios.extend(run[idx_val])
        if real:
            real, rf, gb = np.array(real), np.array(rf), np.array(gb)
            # metricas sobre o empilhado das dobras, nao a media por dobra
            previsoes[col] = {"real": real, "rf": rf, "gb": gb, "run_idx": np.array(ensaios),
                              "metricas": {"rf": metricas(real, rf), "gb": metricas(real, gb)}}
    return previsoes

In [ ]:
def carregar(sufixo):
    ds = {area: pd.read_csv(resolve(PASTA_DADOS, f"dataset_{area}{sufixo}.csv")) for area in AREAS}
    sep = {area: dict(zip(("X", "cols_alvo"), separar(df))) for area, df in ds.items()}
    return ds, sep

PREVISOES, tempos = {}, {}
for sufixo in SUFIXOS:
    datasets, separado = carregar(sufixo)
    PREVISOES[sufixo] = {}
    for area in AREAS:
        t0 = time.perf_counter()
        PREVISOES[sufixo][area] = treinar_area(area)
        tempos[(sufixo, area)] = time.perf_counter() - t0

In [ ]:
pd.DataFrame({
    (sufixo, area): {"componentes": len(PREVISOES[sufixo][area]), "tempo_s": round(tempos[(sufixo, area)], 1)}
    for sufixo in PREVISOES for area in AREAS
}).T

In [ ]:
sufixo, area = SUFIXOS[0], AREAS[0]
componente = next(iter(PREVISOES[sufixo][area]))
print(sufixo, area, componente)
pd.DataFrame(PREVISOES[sufixo][area][componente]["metricas"]).T

In [ ]:
linhas = []
for sufixo, areas in PREVISOES.items():
    for area, comps in areas.items():
        for col, p in comps.items():
            linhas.append(pd.DataFrame({
                "sufixo": sufixo, "area": area, "componente": col,
                "run_idx": p["run_idx"], "real": p["real"], "rf": p["rf"], "gb": p["gb"],
            }))
previsoes = pd.concat(linhas, ignore_index=True)

# grava ao lado dos datasets, onde o resolve() dos notebooks seguintes procura
pasta = os.path.dirname(resolve(PASTA_DADOS, "limites_ativos.csv"))
previsoes.to_csv(os.path.join(pasta, "previsoes_rf_gb.csv"), index=False)
previsoes.shape